In [1]:
!pip install pypdf

In [2]:
import os

# Create the directory if it doesn't exist
data_dir = "/content/KPMG data"
os.makedirs(data_dir, exist_ok=True)
print(f"Created directory: {data_dir}")
print("Please upload your PDFs to this directory using the Files tab on the left.")

Created directory: /content/KPMG data
Please upload your PDFs to this directory using the Files tab on the left.


In [3]:
import argparse
import json
import re
import sqlite3
from pathlib import Path

from pypdf import PdfReader


# Common section names found in research papers.
SECTION_NAMES = [
    "abstract",
    "introduction",
    "related work",
    "background",
    "method",
    "methodology",
    "approach",
    "experiment",
    "experiments",
    "results",
    "evaluation",
    "discussion",
    "conclusion",
    "conclusions",
    "future work",
    "limitations"
]


# Words that may help identify useful AI/business trends.
TREND_WORDS = [
    "state-of-the-art",
    "novel",
    "outperform",
    "benchmark",
    "improve",
    "efficient",
    "scalable",
    "framework",
    "agent",
    "reasoning",
    "multimodal",
    "fine-tune",
    "open-source",
    "cost",
    "latency",
    "deploy",
    "real-world",
    "adoption",
    "risk",
    "safety",
]


def get_text(pdf_file):
    """Read all text from a PDF."""
    reader = PdfReader(str(pdf_file))

    pages = []

    for page in reader.pages:
        pages.append(page.extract_text() or "")

    return "\n".join(pages)


def guess_title(text):
    """Use the first reasonable-looking line as a backup title."""
    for line in text.splitlines():
        line = line.strip()

        if (
            len(line) > 15
            and "arxiv" not in line.lower()
            and not re.search(r"\d{4}\.\d{4,5}", line)
        ):
            return line

    return "Unknown title"


def get_metadata(pdf_file, text):
    """Get title and authors from the PDF metadata when possible."""
    reader = PdfReader(str(pdf_file))
    metadata = reader.metadata or {}

    title = (metadata.title or "").strip()
    authors = (metadata.author or "").strip()

    if not title:
        title = guess_title(text)

    if not authors:
        authors = None

    return title, authors


def find_arxiv_id(text, filename):
    """Find an arXiv ID in the text or filename."""
    match = re.search(r"\b\d{4}\.\d{4,5}(v\d+)?\b", text[:3000])

    if match:
        return match.group(0)

    # Some downloaded filenames use an underscore instead of a period.
    match = re.search(r"(\d{4})_(\d{4,5})(v\d+)?", filename)

    if match:
        version = match.group(3) or ""
        return f"{match.group(1)}.{match.group(2)}{version}"

    return None


def merge_section_headers(text):
    """
    Sometimes a PDF extracts a heading like:

        1
        Introduction

    Turn that into "1 Introduction".
    """
    lines = text.splitlines()
    new_lines = []

    i = 0

    while i < len(lines):
        line = lines[i].strip()

        if re.fullmatch(r"\d{1,2}", line):
            j = i + 1

            while j < len(lines) and not lines[j].strip():
                j += 1

            if j < len(lines) and lines[j].strip().lower() in SECTION_NAMES:
                new_lines.append(line + " " + lines[j].strip())
                i = j + 1
                continue

        new_lines.append(lines[i])
        i += 1

    return "\n".join(new_lines)


def split_sections(text):
    """Split the paper into sections using common section headings."""
    text = merge_section_headers(text)

    names = "|".join(re.escape(name) for name in SECTION_NAMES)

    pattern = re.compile(
        r"(?im)^\s*\d{0,2}\.?\s*(" + names + r")\s*$"
    )

    matches = list(pattern.finditer(text))
    sections = {}

    if matches:
        front_matter = text[:matches[0].start()]
    else:
        front_matter = text

    for i, match in enumerate(matches):
        section_name = match.group(1).lower()
        start = match.end()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(text)

        section_text = text[start:end].strip()

        # Combine repeated section names if a paper has them.
        sections[section_name] = (
            sections.get(section_name, "") + section_text + "\n"
        )

    return sections, front_matter


def looks_like_abstract(line):
    """Check whether a line looks like normal paragraph text."""
    words = line.split()

    if len(words) < 8:
        return False

    lowercase_words = sum(
        1 for word in words if word[:1].islower()
    )

    return lowercase_words / len(words) > 0.5


def get_abstract(sections, front_matter):
    """Get the labeled abstract or make a simple guess."""
    if sections.get("abstract"):
        return sections["abstract"][:1800].strip()

    lines = front_matter.splitlines()

    for i, line in enumerate(lines):
        if looks_like_abstract(line.strip()):
            return "\n".join(lines[i:])[:1800].strip()

    return front_matter[-1800:].strip()


def get_key_sentences(text, max_sentences=5):
    """
    Return a few sentences containing trend-related words.

    This is only a simple filter. A later LLM step can decide
    which claims are actually important.
    """
    sentences = re.split(r"(?<=[.!?])\s+", text)
    key_sentences = []

    for sentence in sentences:
        sentence = sentence.strip().replace("\n", " ")

        if 30 < len(sentence) < 400:
            sentence_lower = sentence.lower()

            if any(word in sentence_lower for word in TREND_WORDS):
                key_sentences.append(sentence)

        if len(key_sentences) >= max_sentences:
            break

    return key_sentences


def split_into_chunks(text, chunk_size=1000):
    """
    Break text into smaller pieces for a future RAG/embedding step.

    Keeping this character-based makes the first version easy to understand.
    """
    words = text.split()
    chunks = []
    current_chunk = []
    current_length = 0

    for word in words:
        if current_length + len(word) + 1 > chunk_size:
            if current_chunk:
                chunks.append(" ".join(current_chunk))

            current_chunk = [word]
            current_length = len(word)
        else:
            current_chunk.append(word)
            current_length += len(word) + 1

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


def process_paper(pdf_file):
    """Extract info we want from one paper."""
    text = get_text(pdf_file)
    sections, front_matter = split_sections(text)

    title, authors = get_metadata(pdf_file, text)

    conclusion = sections.get("conclusion", "")

    if not conclusion:
        conclusion = sections.get("conclusions", "")

    results = sections.get("results", "")

    if not results:
        results = sections.get("evaluation", "")

    methods = sections.get("method", "")

    if not methods:
        methods = sections.get("methodology", "")

    if not methods:
        methods = sections.get("approach", "")

    paper = {
        "file": pdf_file.name,
        "arxiv_id": find_arxiv_id(text, pdf_file.name),
        "title": title,
        "authors": authors,
        "abstract": get_abstract(sections, front_matter),
        "key_findings": get_key_sentences(
            conclusion + " " + results
        ),
        "methodology_snippet": methods[:800].strip(),
        "trend_signals": get_key_sentences(text, max_sentences=8),
        "sections": sections,
        "chunks": split_into_chunks(text),
    }

    return paper


def create_database(db_file):
    """Create the SQLite database and its tables."""
    connection = sqlite3.connect(db_file)

    cursor = connection.cursor()

    # Main information about each paper.
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS papers (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            file_name TEXT,
            arxiv_id TEXT,
            title TEXT,
            authors TEXT,
            abstract TEXT,
            methodology TEXT
        )
        """
    )

    # Store the actual paper sections.
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS sections (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            paper_id INTEGER,
            section_name TEXT,
            section_text TEXT,
            FOREIGN KEY (paper_id) REFERENCES papers(id)
        )
        """
    )

    # Store smaller pieces of text that can later be embedded
    # and placed in ChromaDB or another vector database.
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS chunks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            paper_id INTEGER,
            chunk_number INTEGER,
            chunk_text TEXT,
            FOREIGN KEY (paper_id) REFERENCES papers(id)
        )
        """
    )

    connection.commit()

    return connection


def save_paper(connection, paper):
    """Save one paper and its sections/chunks into SQLite."""
    cursor = connection.cursor()

    cursor.execute(
        """
        INSERT INTO papers
        (file_name, arxiv_id, title, authors, abstract, methodology)
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            paper["file"],
            paper["arxiv_id"],
            paper["title"],
            paper["authors"],
            paper["abstract"],
            paper["methodology_snippet"],
        ),
    )

    paper_id = cursor.lastrowid

    # Save sections.
    for section_name, section_text in paper["sections"].items():
        cursor.execute(
            """
            INSERT INTO sections
            (paper_id, section_name, section_text)
            VALUES (?, ?, ?)
            """,
            (paper_id, section_name, section_text.strip()),
        )

    # Save text chunks for future retrieval.
    for number, chunk in enumerate(paper["chunks"]):
        cursor.execute(
            """
            INSERT INTO chunks
            (paper_id, chunk_number, chunk_text)
            VALUES (?, ?, ?)
            """,
            (paper_id, number, chunk),
        )

    connection.commit()


def make_business_prompt(paper):
    """Create a prompt for the next LLM/business-insight step."""
    return f"""You are a business analyst summarizing AI research for a KPMG audience.

Paper title: {paper["title"]}
Abstract: {paper["abstract"]}
Key findings: {paper["key_findings"]}
Methodology: {paper["methodology_snippet"]}

In 3-4 sentences, explain:
1. What AI capability or trend this paper shows.
2. Why it matters for a business (e.g. efficiency, risk, cost, or new capability).
3. One essential way a professional services firm like KPMG could apply or watch this trend.

Avoid unnecessary technical jargon and write for a non-technical stakeholder.
"""


def main(data_dir, number_of_papers, database_file, json_file):
    """Run the whole extraction pipeline."""
    data_path = Path(data_dir)

    pdf_files = sorted(data_path.glob("*.pdf"))[:number_of_papers]

    if not pdf_files:
        raise SystemExit(f"No PDF files found in {data_dir}")

    connection = create_database(database_file)
    papers = []

    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")

        paper = process_paper(pdf_file)
        paper["business_insight_prompt"] = make_business_prompt(paper)

        save_paper(connection, paper)

        papers.append(paper)

    connection.close()

    # Save a simple JSON dataset too.
    json_path = Path(json_file)
    json_data = []

    for paper in papers:
        json_data.append(
            {
                "file": paper["file"],
                "arxiv_id": paper["arxiv_id"],
                "title": paper["title"],
                "authors": paper["authors"],
                "abstract": paper["abstract"],
                "key_findings": paper["key_findings"],
                "methodology_snippet": paper["methodology_snippet"],
                "trend_signals": paper["trend_signals"],
            }
        )

    json_path.write_text(
        json.dumps(json_data, indent=2),
        encoding="utf-8",
    )

    print()
    print(f"Saved database: {database_file}")
    print(f"Saved JSON dataset: {json_file}")
    print(f"Processed {len(papers)} papers.")


# Call the main function directly with default arguments
main(
    data_dir="/content/KPMG data",
    number_of_papers=5,
    database_file="/content/papers.db",
    json_file="/content/paper_insights.json",
)

Processing: 2608.20316v1_Pandora's AI Model Routing Box Efficient Allocation with Costly Value Estimation.pdf
Processing: 2608.20318v1_AI4AI-Bench Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Imp.pdf
Processing: 2608.20319v1_Inducing Task Models from Computer-Use Traces.pdf
Processing: 2608.20320v1_An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and We.pdf
Processing: 2608.20331v1_G-CARL Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical R.pdf

Saved database: /content/papers.db
Saved JSON dataset: /content/paper_insights.json
Processed 5 papers.
